In [11]:
import pandas as pd
import requests
from io import StringIO


INDEX_URLS = {
    "NIFTY50": "https://www.niftyindices.com/IndexConstituent/ind_nifty50list.csv",
    "NIFTY100": "https://www.niftyindices.com/IndexConstituent/ind_nifty100list.csv",
    "NIFTY200": "https://www.niftyindices.com/IndexConstituent/ind_nifty200list.csv",
    "NIFTY500": "https://www.niftyindices.com/IndexConstituent/ind_nifty500list.csv",
}


SECTOR_PAGES = {
    "NIFTY_AUTO": "https://www.niftyindices.com/indices/equity/sectoral-indices/nifty-auto",
    "NIFTY_BANK": "https://www.niftyindices.com/indices/equity/sectoral-indices/nifty-bank",
    "NIFTY_CEMENT": "https://www.niftyindices.com/indices/equity/sectoral-indices/nifty-cement",
    "NIFTY_CHEMICALS": "https://www.niftyindices.com/indices/equity/sectoral-indices/nifty-chemicals",
    "NIFTY_FINANCIAL_SERVICES": "https://www.niftyindices.com/indices/equity/sectoral-indices/nifty-financial-services",
    "NIFTY_FINANCIAL_SERVICES_25_50": "https://www.niftyindices.com/indices/equity/sectoral-indices/nifty-financial-services-25-50-index",
    "NIFTY_FINANCIAL_SERVICES_EX_BANK": "https://www.niftyindices.com/indices/equity/sectoral-indices/nifty-financial-services-ex-bank",
    "NIFTY_FMCG": "https://www.niftyindices.com/indices/equity/sectoral-indices/nifty-fmcg",
    "NIFTY_HEALTHCARE": "https://www.niftyindices.com/indices/equity/sectoral-indices/nifty-healthcare-index",
    "NIFTY_IT": "https://www.niftyindices.com/indices/equity/sectoral-indices/nifty-it",
    "NIFTY_MEDIA": "https://www.niftyindices.com/indices/equity/sectoral-indices/nifty-media",
    "NIFTY_METAL": "https://www.niftyindices.com/indices/equity/sectoral-indices/nifty-metal",
    "NIFTY_PHARMA": "https://www.niftyindices.com/indices/equity/sectoral-indices/nifty-pharma",
    "NIFTY_PRIVATE_BANK": "https://www.niftyindices.com/indices/equity/sectoral-indices/nifty-private-bank",
    "NIFTY_PSU_BANK": "https://www.niftyindices.com/indices/equity/sectoral-indices/nifty-psu-bank",
    "NIFTY_REALTY": "https://www.niftyindices.com/indices/equity/sectoral-indices/nifty-realty",
    "NIFTY_REITS_REALTY": "https://www.niftyindices.com/indices/equity/sectoral-indices/nifty-reits-realty",
    "NIFTY_CONSUMER_DURABLES": "https://www.niftyindices.com/indices/equity/sectoral-indices/nifty-consumer-durables-index",
    "NIFTY_OIL_AND_GAS": "https://www.niftyindices.com/indices/equity/sectoral-indices/nifty-oil-and-gas-index",
}


SECTOR_CSV_URLS = {
    "NIFTY_AUTO": "https://www.niftyindices.com/IndexConstituent/ind_niftyautolist.csv",
    "NIFTY_BANK": "https://www.niftyindices.com/IndexConstituent/ind_niftybanklist.csv",
    "NIFTY_CEMENT": "https://www.niftyindices.com/IndexConstituent/ind_NiftyCement_list.csv",
    "NIFTY_CHEMICALS": "https://www.niftyindices.com/IndexConstituent/ind_niftyChemicals_list.csv",
    "NIFTY_FINANCIAL_SERVICES": "https://www.niftyindices.com/IndexConstituent/ind_niftyfinancelist.csv",
    "NIFTY_FINANCIAL_SERVICES_25_50": "https://www.niftyindices.com/IndexConstituent/ind_niftyfinancialservices25-50list.csv",
    "NIFTY_FINANCIAL_SERVICES_EX_BANK": "https://www.niftyindices.com/IndexConstituent/ind_niftyfinancialservicesexbank_list.csv",
    "NIFTY_FMCG": "https://www.niftyindices.com/IndexConstituent/ind_niftyfmcglist.csv",
    "NIFTY_HEALTHCARE": "https://www.niftyindices.com/IndexConstituent/ind_niftyhealthcarelist.csv",
    "NIFTY_IT": "https://www.niftyindices.com/IndexConstituent/ind_niftyitlist.csv",
    "NIFTY_MEDIA": "https://www.niftyindices.com/IndexConstituent/ind_niftymedialist.csv",
    "NIFTY_METAL": "https://www.niftyindices.com/IndexConstituent/ind_niftymetallist.csv",
    "NIFTY_PHARMA": "https://www.niftyindices.com/IndexConstituent/ind_niftypharmalist.csv",
    "NIFTY_PRIVATE_BANK": "https://www.niftyindices.com/IndexConstituent/ind_nifty_privatebanklist.csv",
    "NIFTY_PSU_BANK": "https://www.niftyindices.com/IndexConstituent/ind_niftypsubanklist.csv",
    "NIFTY_REALTY": "https://www.niftyindices.com/IndexConstituent/ind_niftyrealtylist.csv",
    "NIFTY_CONSUMER_DURABLES": "https://www.niftyindices.com/IndexConstituent/ind_niftyconsumerdurableslist.csv",
    "NIFTY_OIL_AND_GAS": "https://www.niftyindices.com/IndexConstituent/ind_niftyoilgaslist.csv",
}


SECTOR_FALLBACK_SYMBOLS = {
    "NIFTY_REITS_REALTY": [
        "NSE:EMBASSY-EQ",
        "NSE:BROOKFIELD-EQ",
        "NSE:NEXUSSELECT-EQ",
        "NSE:KRT-EQ",
        "NSE:MINDSPACE-EQ",
        "NSE:DLF-EQ",
        "NSE:PHOENIXLTD-EQ",
        "NSE:GODREJPROP-EQ",
        "NSE:LODHA-EQ",
        "NSE:PRESTIGE-EQ",
        "NSE:OBEROIRLTY-EQ",
        "NSE:BRIGADE-EQ",
        "NSE:ANANTRAJ-EQ",
        "NSE:SOBHA-EQ",
        "NSE:SIGNATURE-EQ",
    ],
}


def _headers() -> dict[str, str]:
    return {
        "User-Agent": "Mozilla/5.0",
        "Referer": "https://www.niftyindices.com/",
        "Accept": "text/csv,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    }


def _to_fyers(symbols: list[str]) -> list[str]:
    return [f"NSE:{symbol}-EQ" for symbol in symbols]


def _from_fyers(symbols: list[str]) -> list[str]:
    clean = []
    for symbol in symbols:
        s = str(symbol)
        if s.startswith("NSE:"):
            s = s[4:]
        if s.endswith("-EQ"):
            s = s[:-3]
        clean.append(s)
    return clean


def _read_constituent_csv(csv_url: str) -> pd.DataFrame:
    response = requests.get(csv_url, headers=_headers(), timeout=30)
    response.raise_for_status()

    df = pd.read_csv(StringIO(response.text))

    if "Symbol" not in df.columns:
        raise RuntimeError(f"'Symbol' column missing in CSV: {csv_url}")

    return df


def get_all_sector_names() -> list[str]:
    return list(SECTOR_PAGES.keys())


def load_index_constituents(index_name: str, fyers_format: bool = True) -> list[str]:
    index_key = index_name.upper()
    csv_url = INDEX_URLS.get(index_key)

    if not csv_url:
        print(f"[WARN] Unsupported index: {index_name}")
        return []

    try:
        df = _read_constituent_csv(csv_url)
        symbols = df["Symbol"].dropna().astype(str).str.strip().tolist()
        return _to_fyers(symbols) if fyers_format else symbols
    except Exception as e:
        print(f"[ERROR] {index_name}: {e}")
        return []


def load_sector_constituents(sector_name: str, fyers_format: bool = True) -> list[str]:
    sector_key = sector_name.upper()

    fallback_symbols = SECTOR_FALLBACK_SYMBOLS.get(sector_key)
    if fallback_symbols is not None:
        return fallback_symbols if fyers_format else _from_fyers(fallback_symbols)

    csv_url = SECTOR_CSV_URLS.get(sector_key)
    if not csv_url:
        print(f"[WARN] Unsupported sector or missing CSV: {sector_name}")
        return []

    try:
        df = _read_constituent_csv(csv_url)
        symbols = df["Symbol"].dropna().astype(str).str.strip().tolist()
        return _to_fyers(symbols) if fyers_format else symbols
    except Exception as e:
        print(f"[ERROR] {sector_name}: {e}")
        return []


def load_all_sectors(fyers_format: bool = True) -> dict[str, list[str]]:
    result = {}
    for sector_name in SECTOR_PAGES:
        result[sector_name] = load_sector_constituents(
            sector_name,
            fyers_format=fyers_format,
        )
    return result


def load_all_sectors_df(fyers_format: bool = True) -> pd.DataFrame:
    rows = []

    for sector_name, symbols in load_all_sectors(fyers_format=fyers_format).items():
        for symbol in symbols:
            rows.append(
                {
                    "sector": sector_name,
                    "symbol": symbol,
                }
            )

    return pd.DataFrame(rows)

In [12]:
symbols = load_index_constituents("NIFTY200")
print(len(symbols))
print(symbols[:10])

200
['NSE:360ONE-EQ', 'NSE:ABB-EQ', 'NSE:APLAPOLLO-EQ', 'NSE:AUBANK-EQ', 'NSE:ADANIENSOL-EQ', 'NSE:ADANIENT-EQ', 'NSE:ADANIGREEN-EQ', 'NSE:ADANIPORTS-EQ', 'NSE:ADANIPOWER-EQ', 'NSE:ATGL-EQ']


In [13]:
bank_symbols = load_sector_constituents("NIFTY_BANK")
print(len(bank_symbols))
print(bank_symbols[:10])

14
['NSE:AUBANK-EQ', 'NSE:AXISBANK-EQ', 'NSE:BANKBARODA-EQ', 'NSE:CANBK-EQ', 'NSE:FEDERALBNK-EQ', 'NSE:HDFCBANK-EQ', 'NSE:ICICIBANK-EQ', 'NSE:IDFCFIRSTB-EQ', 'NSE:INDUSINDBK-EQ', 'NSE:KOTAKBANK-EQ']


In [14]:
all_sectors = load_all_sectors()
for sector, symbols in all_sectors.items():
    print(sector, len(symbols))

NIFTY_AUTO 15
NIFTY_BANK 14
NIFTY_CEMENT 16
NIFTY_CHEMICALS 20
NIFTY_FINANCIAL_SERVICES 20
NIFTY_FINANCIAL_SERVICES_25_50 20
NIFTY_FINANCIAL_SERVICES_EX_BANK 30
NIFTY_FMCG 15
NIFTY_HEALTHCARE 20
NIFTY_IT 10
NIFTY_MEDIA 10
NIFTY_METAL 15
NIFTY_PHARMA 20
NIFTY_PRIVATE_BANK 10
NIFTY_PSU_BANK 12
NIFTY_REALTY 10
NIFTY_REITS_REALTY 15
NIFTY_CONSUMER_DURABLES 13
NIFTY_OIL_AND_GAS 15


In [15]:
sector_df = load_all_sectors_df()
print(sector_df.head())
print(sector_df["sector"].value_counts())

       sector             symbol
0  NIFTY_AUTO    NSE:ASHOKLEY-EQ
1  NIFTY_AUTO  NSE:BAJAJ-AUTO-EQ
2  NIFTY_AUTO  NSE:BHARATFORG-EQ
3  NIFTY_AUTO    NSE:BOSCHLTD-EQ
4  NIFTY_AUTO   NSE:EICHERMOT-EQ
sector
NIFTY_FINANCIAL_SERVICES_EX_BANK    30
NIFTY_CHEMICALS                     20
NIFTY_FINANCIAL_SERVICES            20
NIFTY_FINANCIAL_SERVICES_25_50      20
NIFTY_HEALTHCARE                    20
NIFTY_PHARMA                        20
NIFTY_CEMENT                        16
NIFTY_AUTO                          15
NIFTY_FMCG                          15
NIFTY_METAL                         15
NIFTY_REITS_REALTY                  15
NIFTY_OIL_AND_GAS                   15
NIFTY_BANK                          14
NIFTY_CONSUMER_DURABLES             13
NIFTY_PSU_BANK                      12
NIFTY_IT                            10
NIFTY_MEDIA                         10
NIFTY_PRIVATE_BANK                  10
NIFTY_REALTY                        10
Name: count, dtype: int64
